### Modelling

#### Train and evaluate 3 Machine Learning models for our London property prices

#### Importing Packages and Data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

In [2]:
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
np.random.seed(42)

In [4]:
project_root = Path(".").resolve().parent
features_path = project_root / "data" / "processed" / "london_features.parquet"
model_path = project_root / "models" / "best_model.joblib"

In [5]:
df = pd.read_parquet(features_path)
df

,price,date_of_transfer,postcode,property_type,old_new,duration,district,postcode_district,property_type_ordinal,district_sales_count,log_price,latitude,longitude,london_zone,distance_to_station_km,is_central
0,215000,2025-01-31,E3 2PQ,F,N,L,TOWER HAMLETS,E3,1,530,5.332438,51.530123,-0.016067,2.0,0.414205,1
1,349000,2025-12-22,E17 7LB,F,N,L,WALTHAM FOREST,E17,1,1046,5.542825,51.582287,-0.028159,3.0,0.309116,0
2,540000,2025-12-16,NW4 3PG,S,N,F,BARNET,NW4,3,212,5.732394,51.577269,-0.230011,3.0,0.671918,0
3,460000,2025-12-15,E2 8FZ,F,N,L,HACKNEY,E2,1,330,5.662758,51.534910,-0.074545,1.0,0.385880,1
4,265000,2025-12-19,E14 9BF,F,N,L,TOWER HAMLETS,E14,1,879,5.423246,51.506823,-0.005480,2.0,0.165213,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72587,393800,2025-06-11,E10 7PE,F,N,L,WALTHAM FOREST,E10,1,450,5.595276,51.567063,-0.030880,3.0,0.403633,0
72588,388000,2025-05-29,RM13 9SN,S,N,F,HAVERING,RM13,3,314,5.588832,51.525933,0.219288,6.0,2.195610,0
72589,358000,2025-06-05,RM3 7DX,T,N,F,HAVERING,RM3,2,402,5.553883,51.602708,0.205338,6.0,2.213750,0
72590,1095000,2025-01-03,E18 2PS,S,N,F,REDBRIDGE,E18,3,226,6.039414,51.599639,0.014872,4.0,1.273380,0


#### Features and Target

In [6]:
# Target Variable: Log Price
TARGET = "log_price"

# Features
NUMERIC_FEATURES = [
    "property_type_ordinal",
    "district_sales_count",
    "latitude",
    "longitude",
    "london_zone",
    "distance_to_station_km",
    "is_central"
]

LOW_CATEGORICAL = [ # These features have just a few unique values
    "property_type",
    "old_new",
    "duration"
]

HIGH_CATEGORICAL = [ # Many values
    "district",
    "postcode_district"
]

ALL_FEATURES = NUMERIC_FEATURES + LOW_CATEGORICAL + HIGH_CATEGORICAL

#### Train/Test Split

In [7]:
X = df[ALL_FEATURES]
y = df[TARGET]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
X_train

,property_type_ordinal,district_sales_count,latitude,longitude,london_zone,distance_to_station_km,is_central,property_type,old_new,duration,district,postcode_district
19484,1,332,51.477344,-0.326956,4.0,0.460050,0,F,N,L,HOUNSLOW,TW7
53204,2,540,51.566835,-0.400333,5.0,0.983299,0,T,N,F,HILLINGDON,HA4
68793,1,879,51.498458,-0.019187,2.0,0.271989,1,F,N,L,TOWER HAMLETS,E14
24745,2,286,51.396051,-0.123207,3.0,1.220230,0,T,N,F,CROYDON,CR7
4724,4,270,51.406039,0.078947,5.0,1.488010,0,D,N,F,BROMLEY,BR7
...,...,...,...,...,...,...,...,...,...,...,...,...
37194,2,450,51.572768,-0.005523,3.0,0.380674,0,T,N,F,WALTHAM FOREST,E10
6265,2,109,51.619011,-0.073604,4.0,0.424578,0,T,N,F,ENFIELD,N18
54886,1,77,51.462803,-0.389217,5.0,1.170260,0,F,N,L,HOUNSLOW,TW4
860,2,988,51.460531,-0.199595,2.0,0.796135,1,T,N,F,WANDSWORTH,SW18


In [10]:
X_test

,property_type_ordinal,district_sales_count,latitude,longitude,london_zone,distance_to_station_km,is_central,property_type,old_new,duration,district,postcode_district
58084,3,109,51.580014,0.096159,4.0,0.632390,0,S,N,F,REDBRIDGE,IG2
29210,1,248,51.411600,-0.280925,5.0,0.229559,0,F,N,L,KINGSTON UPON THAMES,KT1
26805,2,375,51.399110,-0.126632,3.0,1.368180,0,T,N,F,MERTON,CR4
64103,1,505,51.478383,-0.009487,2.0,0.268088,1,F,N,L,GREENWICH,SE10
51111,2,396,51.650885,-0.219079,5.0,1.719960,0,T,N,F,BARNET,EN5
...,...,...,...,...,...,...,...,...,...,...,...,...
16234,2,988,51.456544,-0.183630,2.0,0.580274,1,T,N,F,WANDSWORTH,SW18
10009,3,443,51.514342,-0.280432,3.0,0.381203,0,S,N,F,EALING,W3
12788,1,596,51.346460,-0.093354,6.0,0.199991,0,F,N,L,CROYDON,CR2
29890,1,190,51.625713,-0.162881,4.0,1.242970,0,F,N,L,BARNET,N20


In [11]:
y_train.shape

(58073,)

In [12]:
y_test.shape

(14519,)

#### Preprocessing

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("low_cat", OneHotEncoder(handle_unknown="ignore"), LOW_CATEGORICAL),
        ("high_cat", TargetEncoder(random_state=42), HIGH_CATEGORICAL)
    ]
)

In [14]:
X_sample = X_train.head(10)
y_sample = y_train.head(10)

preprocessor.fit(X_sample, y_sample)
X_transformed = preprocessor.transform(X_sample)

In [15]:
X_transformed

array([[-0.86204366, -0.58137735,  0.35279272, -1.27145291,  0.29851116,
        -0.74926867, -0.33333333,  0.        ,  1.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ,
         5.43933269,  5.43933269],
       [ 0.09578263,  0.06632746,  1.74098402, -1.79069706,  1.29354835,
         0.61162148, -0.33333333,  0.        ,  0.        ,  0.        ,
         1.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         5.82282165,  5.82282165],
       [-0.86204366,  1.12196174,  0.68031472,  0.90644006, -1.69156322,
        -1.2383864 ,  3.        ,  0.        ,  1.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ,
         5.62838893,  5.62838893],
       [ 0.09578263, -0.72461976, -0.90823057,  0.17035414, -0.69652603,
         1.22784257, -0.33333333,  0.        ,  0.        ,  0.        ,
         1.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         5.71403686

In [16]:
X_transformed.shape

(10, 17)

#### Prediction in Pounds

In [17]:
# Our price has been log transformed and our model predicts log10(price). To make meaningful insights we need to conver it back into pounds and compute MAE, RMSE and R2 in pounds

def evaluate_in_pounds(y_true_log, y_pred_log):
    y_true = 10 ** y_true_log
    y_pred = 10 ** y_pred_log

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {"MAE_pounds": mae, "RMSE_pounds": rmse, "R2_pounds": r2}

#### Models

In [25]:
models = {
    "Linear Regression (Ridge)": Pipeline([
        ("prep", preprocessor),
        ("model", Ridge(alpha=1)),
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestRegressor(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)),
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("model", XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, n_jobs=-1, random_state=42)),
    ])
}

#### Cross Validation

In [21]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

In [28]:
for name, pipeline in models.items():
    print(f" Cross-validation for {name}")

    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=["neg_mean_absolute_error", "r2"], n_jobs=-1
    )

    mae_log = -scores["test_neg_mean_absolute_error"].mean()
    r2 = scores["test_r2"].mean()
    r2_std = scores["test_r2"].std()

    results.append({
        "Model": name,
        "CV MAE (log10)": round(mae_log, 4),
        "CV R²": round(r2, 4),
        "CV R² std": round(r2_std, 4),
    })
    print(f"CV R² = {r2:.4f} (±{r2_std:.4f})")

 Cross-validation for Linear Regression (Ridge)
CV R² = 0.5778 (±0.0046)
 Cross-validation for Random Forest
CV R² = 0.6882 (±0.0030)
 Cross-validation for XGBoost
CV R² = 0.6708 (±0.0050)


#### Evaluation

In [26]:
best_name = "Random Forest"
best_model = models[best_name]

print(f"Selected model: {best_name}")

best_model.fit(X_train, y_train)


Selected model: Random Forest


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('low_cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [29]:
y_pred_log = best_model.predict(X_test)

metrics = evaluate_in_pounds(y_test, y_pred_log)

print(f"  MAE:  £{metrics['MAE_pounds']:,.0f}")
print(f"  RMSE: £{metrics['RMSE_pounds']:,.0f}")
print(f"  R²:   {metrics['R2_pounds']:.4f}")

  MAE:  £161,266
  RMSE: £362,676
  R²:   0.6058


In [30]:
final_model = models["Random Forest"]
final_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('low_cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [31]:
model_path = project_root / "models" / "random_forest_pipeline.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, model_path)

['C:\\Users\\Abdul Qudus\\Documents\\Data Portfolio\\london-house-price-prediction\\models\\random_forest_pipeline.joblib']